# FakeNews Detection Platform — Train DistilBERT in Google Colab

This notebook is the Colab GPU training handoff for the local React + FastAPI project. It trains a DistilBERT classifier on the included ISOT Fake/True dataset, evaluates it on the held-out random test split, and downloads a model ZIP that can be extracted into `backend/models/research_distilbert_random` on Windows.

**Labels:** `0 = Fake News`, `1 = True News`. Training and evaluation metrics describe this dataset only; they do not prove real-world factual accuracy. Use a Colab GPU runtime. Do not upload `.env` or API keys.


## 1. Upload the project ZIP

Upload the complete project ZIP you downloaded from ChatGPT. It must contain `backend/training/train_distilbert.py` and `data/raw/Fake.csv` / `data/raw/True.csv`.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os, subprocess, sys, json

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise RuntimeError('Upload the complete FakeNews Detection project .zip file.')
PROJECT_ZIP = Path('/content') / zip_names[0]
WORK_DIR = Path('/content/FakeNewsResearchPlatform')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
with zipfile.ZipFile(PROJECT_ZIP) as z:
    z.extractall('/content/project_extract')
roots = list(Path('/content/project_extract').glob('*/backend/training/train_distilbert.py'))
if not roots:
    raise FileNotFoundError('Could not find backend/training/train_distilbert.py inside the uploaded ZIP.')
source_root = roots[0].parents[2]
shutil.copytree(source_root, WORK_DIR, ignore=shutil.ignore_patterns('.venv','venv','node_modules','.env','*.sqlite3','__pycache__','.git'))
os.chdir(WORK_DIR)
assert (WORK_DIR/'data/raw/Fake.csv').exists() and (WORK_DIR/'data/raw/True.csv').exists()
print('Project:', WORK_DIR)
print('ZIP:', PROJECT_ZIP.name)


## 2. Confirm GPU and install dependencies

In Colab choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Change the Colab runtime to GPU and rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK_DIR/'backend'/'requirements-colab.txt')], check=True)


## 3. Prepare clean data and split it

The preparation script removes exact duplicates and conflicting exact texts. The test set remains separate from training and validation.

In [ ]:
def run_module(module, *args):
    cmd = [sys.executable, '-m', module, *map(str, args)]
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=WORK_DIR, check=True)

run_module('backend.training.prepare_dataset')
run_module('backend.training.create_research_splits')
run_module('backend.training.validate_splits')


## 4. Train DistilBERT

Default: two epochs, max token length 256, batch size 8. Training time depends on GPU availability and Colab limits. If a run is interrupted, rerun this cell; the model checkpoint is saved when validation macro-F1 improves.

In [ ]:
MODEL_DIR = WORK_DIR/'backend'/'models'/'research_distilbert_random'
run_module('backend.training.train_distilbert', '--protocol','random', '--epochs','2', '--batch-size','8', '--max-length','256', '--learning-rate','2e-5', '--seed','42', '--output-dir',str(MODEL_DIR))
if not (MODEL_DIR/'config.json').exists():
    raise RuntimeError('Training did not produce config.json. Check the training output above.')
print('Saved model:', MODEL_DIR)


## 5. Evaluate on the untouched random test split

Report the metrics from this cell. Do not describe the result as proof of general real-world accuracy.

In [ ]:
TEST_CSV = WORK_DIR/'data'/'processed'/'research_splits'/'random'/'test.csv'
run_module('backend.training.evaluate_model', '--model-type','transformer', '--model-path',str(MODEL_DIR), '--test-csv',str(TEST_CSV), '--max-length','256', '--batch-size','16', '--output-prefix',str(WORK_DIR/'outputs'/'research'/'evaluation_distilbert_random'))


## 6. Package and download the model

Extract the downloaded model ZIP into your local project's `backend/models/` folder. The final path must be `backend/models/research_distilbert_random/config.json`. Keep `MODEL_DIR=backend/models/research_distilbert_random` in your local `.env`.

In [ ]:
MODEL_ZIP = Path('/content/research_distilbert_random.zip')
if MODEL_ZIP.exists(): MODEL_ZIP.unlink()
with zipfile.ZipFile(MODEL_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in MODEL_DIR.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(MODEL_DIR.parent))
report = WORK_DIR/'outputs'/'research'/'evaluation_distilbert_random.json'
if report.exists():
    files.download(str(report))
files.download(str(MODEL_ZIP))
print('Model ZIP ready:', MODEL_ZIP)


## Local Windows installation after training

1. Extract the downloaded model ZIP into `<project>\backend\models\`.
2. Confirm the resulting path is `backend\models\research_distilbert_random\config.json`.
3. In project-root `.env`, set `MODEL_DIR=backend/models/research_distilbert_random`.
4. Restart the backend. The API will prefer DistilBERT when the directory is valid and fall back to the included TF-IDF baseline if it is absent.
5. On the analysis page, the explanation method should show **Captum Saliency** when the model and Captum are installed.
